In [1]:
# ===== [KAGGLE] bootstrap: recreate the Colab layout =====
# The paper notebooks expect /content/drive/MyDrive/bsm_paper_replication and
# /content/BSM_Project_2. bsm_env builds both from the attached dataset; the
# pipeline itself (images, augmentation, splits, training) is unchanged.
import pathlib, sys
_code = pathlib.Path("/kaggle/input/bsm-paper-2-kaggle/code")
if not _code.is_dir():
    _hits = list(pathlib.Path("/kaggle/input").rglob("bsm_env.py"))
    _code = _hits[0].parent if _hits else _code
if str(_code) not in sys.path:
    sys.path.insert(0, str(_code))
import bsm_env
BSM_RAW_SRC = bsm_env.setup()


[bsm_env] using existing /content
[bsm_env] mode=existing  project=/content/drive/MyDrive/bsm_paper_replication
[bsm_env] raw images=367  originals=367
[bsm_env] models.py: ok
[bsm_env] common.py: ok
[bsm_env] torch 2.11.0+cu128 cuda=True Tesla T4
[bsm_env] disk /kaggle/working free=20.9 GB


# BSM Paper Replication — Notebook 09: comparison of the four architectures

Collects the four paper-faithful runs (notebooks 05-08) into the paper's Table 9 layout
(accuracy / precision / recall) and additionally evaluates every model on the paper's
"18 untrained images, 3 per class" subset, sampled from the test split with seed 42
(DEVIATIONS row 6).

The paper's own Table 9 numbers are printed in clearly labelled `PAPER ...` columns and are
never mixed with the measured values of this repository.


In [2]:
# ===== faithful comparison: Table 9 layout + paper's 18-image test subset =====
import os, sys, json, pathlib
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

PROJECT = pathlib.Path("/content/drive/MyDrive/bsm_paper_replication")
VER = PROJECT / "versions"; CK = PROJECT / "checkpoints"; RES = PROJECT / "results"
FIG = PROJECT / "figures"; FIG.mkdir(parents=True, exist_ok=True)
if str(PROJECT) not in sys.path: sys.path.insert(0, str(PROJECT))
from models import dresnet50, resnet18, resnet34, vgg16
import common

CLASSES = ["Ilmenite","Rutile","Zircon","Garnet","Monazite","Sillimanite"]
MODELS = [("dresnet50", dresnet50, "D-ResNet-50", 0.89, 0.90, 0.89),
          ("resnet18",  resnet18,  "ResNet-18",  0.78, 0.87, 0.78),
          ("resnet34",  resnet34,  "ResNet-34",  0.83, 0.89, 0.83),
          ("vgg16",     vgg16,     "VGG16",      0.83, 0.89, 0.83)]

def load_xy(prefix):
    X = np.load(VER / f"X_{prefix}.npy"); y = np.load(VER / f"y_{prefix}.npy")
    return torch.from_numpy(X.transpose(0,3,1,2)).contiguous(), torch.from_numpy(y.astype(np.int64))

Xte, yte = load_xy("test")
device = torch.device("cuda")
test_loader = DataLoader(TensorDataset(Xte, yte), batch_size=32, shuffle=False)

# ---- the paper's "18 unseen images" (3 per class) sampled from the test split, seed 42
rng = np.random.default_rng(42)
yte_np = yte.numpy()
sel = []
for c in range(6):
    idx = np.where(yte_np == c)[0]
    sel += sorted(rng.choice(idx, size=3, replace=False).tolist())
sel = np.array(sel)
print("paper-18 subset (indices into the test split):", sel.tolist())
print("subset class counts:", {CLASSES[i]: int((yte_np[sel] == i).sum()) for i in range(6)})

rows = []
cms = {}
sub_rows = []
for key, builder, pretty, p_acc, p_prec, p_rec in MODELS:
    ck_path = CK / key / f"{key}_best.pt"
    mpath = RES / f"metrics_{key}.json"
    if not ck_path.exists():
        print("MISSING checkpoint for", key, "- run the matching notebook first")
        continue
    model = builder().to(device)
    model.load_state_dict(torch.load(ck_path, map_location=device, weights_only=True))
    model.eval()
    preds = []
    with torch.no_grad():
        for xb, yb in test_loader:
            preds.append(model(xb.to(device)).argmax(1).cpu().numpy())
    y_pred = np.concatenate(preds); y_true = yte_np
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=range(6)); cms[pretty] = cm
    spec = float(np.mean([(cm.sum()-cm[i,:].sum()-cm[:,i].sum()+cm[i,i])/(cm.sum()-cm[i,:].sum()) for i in range(6)]))
    best_epoch = -1
    if mpath.exists():
        best_epoch = json.load(open(mpath)).get("best_epoch", -1)
    rows.append({"Model": pretty, "Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1,
                 "Specificity": spec, "Best epoch": best_epoch, "n test": int(len(y_true)),
                 "PAPER Accuracy": p_acc, "PAPER Precision": p_prec, "PAPER Recall": p_rec})
    # 18-image subset
    ys_, ps_ = y_true[sel], y_pred[sel]
    a18 = accuracy_score(ys_, ps_)
    p18, r18, f18, _ = precision_recall_fscore_support(ys_, ps_, average="macro", zero_division=0)
    sub_rows.append({"Model": pretty, "18-img accuracy": a18, "18-img macro P": p18,
                     "18-img macro R": r18, "18-img macro F1": f18,
                     "correct": int((ys_ == ps_).sum()), "of": int(len(ys_))})
    print(f"{pretty}: test acc {acc:.4f} | 18-img acc {a18:.4f} ({int((ys_==ps_).sum())}/18)")

df = pd.DataFrame(rows)
print("\n=== FAITHFUL RUNS (measured) vs PAPER Table 9 (reference only) ===")
print(df.to_string(index=False))
if sub_rows:
    df18 = pd.DataFrame(sub_rows)
    print("\n=== paper's 18 unseen images (3 per class, sampled from the test split, seed 42) ===")
    print(df18.to_string(index=False))

# ---- per-class table for the best model (by accuracy)
if cms:
    best = max(cms, key=lambda k: df.loc[df["Model"] == k, "Accuracy"].iloc[0])
    print(f"\nper-class metrics — {best}:")
    print(common.metrics_from_cm(cms[best], CLASSES).to_string(index=False))

    fig, axes = plt.subplots(1, len(cms), figsize=(4*len(cms), 3.8))
    if len(cms) == 1: axes = [axes]
    for ax, (pretty, cm) in zip(axes, cms.items()):
        im = ax.imshow(cm, cmap="Blues")
        ax.set_xticks(range(6), CLASSES, rotation=45, ha="right", fontsize=7)
        ax.set_yticks(range(6), CLASSES, fontsize=7)
        ax.set_title(pretty, fontsize=9)
        for i in range(6):
            for j in range(6):
                ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=6,
                        color="white" if cm[i, j] > cm.max()/2 else "black")
    fig.colorbar(im, ax=axes, fraction=0.02); fig.tight_layout()
    fig.savefig(FIG / "confusion_matrices_faithful.png", dpi=150)
    print("saved figure", FIG / "confusion_matrices_faithful.png")

# ---- training curves for every model that has a history file
for key, builder, pretty, *_ in MODELS:
    hpath = RES / f"history_{key}.json"
    if not hpath.exists(): continue
    hist = json.load(open(hpath))
    common.plot_loss_curves(hist, save_to=FIG / f"loss_{key}.png")
    common.plot_accuracy_curves(hist, save_to=FIG / f"accuracy_{key}.png")
    print("saved curves for", pretty)

out = {"faithful_measured": df.to_dict("records"),
       "paper_table9_reference_only": {k: {"accuracy": a, "precision": p, "recall": r} for k, _, _, a, p, r in MODELS},
       "paper_18_subset": sub_rows,
       "paper_18_indices": sel.tolist()}
with open(RES / "comparison_faithful.json", "w") as f:
    json.dump(out, f, indent=2, default=str)
print("\nsaved", RES / "comparison_faithful.json")
print("COMPARISON DONE")


paper-18 subset (indices into the test split): [126, 797, 930, 142, 842, 1029, 578, 903, 1178, 218, 612, 906, 197, 488, 637, 470, 805, 974]
subset class counts: {'Ilmenite': 3, 'Rutile': 3, 'Zircon': 3, 'Garnet': 3, 'Monazite': 3, 'Sillimanite': 3}
D-ResNet-50: test acc 0.7583 | 18-img acc 0.6667 (12/18)
ResNet-18: test acc 0.9142 | 18-img acc 0.9444 (17/18)
ResNet-34: test acc 0.7673 | 18-img acc 0.7778 (14/18)
VGG16: test acc 0.2607 | 18-img acc 0.2222 (4/18)

=== FAITHFUL RUNS (measured) vs PAPER Table 9 (reference only) ===
      Model  Accuracy  Precision   Recall       F1  Specificity  Best epoch  n test  PAPER Accuracy  PAPER Precision  PAPER Recall
D-ResNet-50  0.758251   0.758672 0.757268 0.756101     0.951751          60    1212            0.89             0.90          0.89
  ResNet-18  0.914191   0.921295 0.913581 0.914928     0.982757          60    1212            0.78             0.87          0.78
  ResNet-34  0.767327   0.783571 0.770408 0.764486     0.953601          

/tmp/ipykernel_320/2578365934.py:105: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.colorbar(im, ax=axes, fraction=0.02); fig.tight_layout()


saved figure /content/drive/MyDrive/bsm_paper_replication/figures/confusion_matrices_faithful.png
saved curves for D-ResNet-50
saved curves for ResNet-18
saved curves for ResNet-34
saved curves for VGG16

saved /content/drive/MyDrive/bsm_paper_replication/results/comparison_faithful.json
COMPARISON DONE


In [3]:
# persist notebook copy
import glob, os, shutil
PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
os.makedirs(os.path.join(PROJECT,"notebooks"), exist_ok=True)
cands = glob.glob("/content/drive/MyDrive/Colab Notebooks/*.ipynb")
if cands:
    newest = max(cands, key=os.path.getmtime)
    shutil.copy(newest, os.path.join(PROJECT,"notebooks","09_comparison.ipynb"))
    print("copied ->", os.path.join(PROJECT,"notebooks","09_comparison.ipynb"))
